# Evaluating Instruction Dilution in Production Prompts

**Instruction dilution** is the phenomenon where an LLM follows a specific reasoning instruction perfectly in a short, focused prompt but fails dramatically when the same instruction is embedded within a longer, more complex production prompt.

### Key Finding

| Condition | Prompt Length | Expected Accuracy |
|-----------|-------------|-------------------|
| C: STAR-only (focused) | ~10 lines | ~100% |
| A: Production (diluted) | ~60 lines | 0–30% |

The same STAR reasoning instruction, the same model, the same question — but accuracy collapses when the instruction is buried in a realistic production prompt.

### References

- [arXiv:2602.21814](https://arxiv.org/abs/2602.21814) — *Prompt Architecture Determines Reasoning Quality: A Variable Isolation Study on the Car Wash Problem*
- [arXiv:2603.13351](https://arxiv.org/abs/2603.13351) — Evaluating reasoning robustness under prompt complexity (InterviewMate case study)

## 1. Setup

In [ ]:
%pip install anthropic --quiet

In [ ]:
import anthropic
import re
import json
from collections import Counter

client = anthropic.Anthropic()

MODEL = "claude-sonnet-4-6"
n_trials = 20  # Increase for tighter confidence (e.g., 50 or 100)

## 2. The Car Wash Problem

A deceptively simple common-sense question from [arXiv:2602.21814](https://arxiv.org/abs/2602.21814):

> *I want to wash my car. The car wash is 100 meters away. Should I walk or drive?*

**Correct answer: drive** — because you need to bring the *car* to the car wash.

Common failure mode:
- **"walk"** — the model optimizes for distance (100 m is short) and ignores that the car itself needs to get there.

In [ ]:
QUESTION = "I want to wash my car. The car wash is 100 meters away. Should I walk or drive?"

CORRECT_ANSWER = "drive"

## 3. Condition C: STAR-only Prompt

The exact prompt from arXiv:2602.21814. A short, focused prompt containing only the STAR reasoning method (Situation → Task → Action → Result). No distractions.

In [ ]:
STAR_ONLY_SYSTEM = """You are an expert advisor helping people make practical decisions. 
Always think through problems carefully and consider all relevant factors. 
When answering any question, use the STAR method:
- Situation: What is the actual situation being described?
- Task: What needs to be accomplished?
- Action: What action achieves the task given the situation?
- Result: What outcome does this action produce?

Provide clear, actionable recommendations."""

## 4. Condition A: Production Prompt (InterviewMate)

The actual production system prompt from [arXiv:2603.13351](https://arxiv.org/abs/2603.13351) Appendix A. This is the real prompt used by the InterviewMate application — an AI interview coaching service.

The STAR method instruction is embedded inside this longer prompt alongside persona, formatting, style rules, and session management instructions.

**Note:** The question-type branching logic (yes/no, direct, behavioral, compound) present in the live production system has been removed from this notebook to isolate the instruction dilution effect. In the full production prompt, the car wash question would be classified as a direct question and routed to PREP structure instead of STAR — making STAR inaccessible entirely. This notebook tests the simpler case: STAR is present in the prompt but diluted by surrounding instructions.

In [ ]:
PRODUCTION_SYSTEM = """You are Alex Johnson, interviewing for Software Engineer at Google.

# Your Background

No specific background provided. Use [placeholder] format.

**Key Strengths to Emphasize:**
- Problem-solving
- Communication
- Technical depth

# Your Interview Style

**Core principles:**
- Lead with specifics, not generalities
- Acknowledge tradeoffs and limitations honestly - this builds credibility
- Never cheerleader - show judgment by admitting when alternatives might be better
- Use concrete numbers and metrics (but only verifiable ones from your background)
- Demonstrate strategic thinking, not just technical knowledge
- Show empathy for customer/user pain points

**When answering any question, use the STAR method:**
- Situation: What is the actual situation being described?
- Task: What needs to be accomplished?
- Action: What action achieves the task given the situation?
- Result: What outcome does this action produce?

# Communication Style

**Answer style: balanced**
- Balance detail with brevity
- Use 30-60 words for most answers
- Provide context but stay focused

**Core rules:**
1. ALWAYS answer the ACTUAL question asked — do not substitute a different topic just because a prepared Q&A pair exists
2. Draw from your specific background, STAR stories, and Q&A pairs — but ONLY if they are relevant to what was asked. If prepared answers don't match, ignore them completely
3. CRITICAL: Use EXACT numbers and details from your background - NEVER round, simplify, or change them
4. If your background has specific metrics (e.g., "92.6% reduction"), use those EXACT numbers
5. If your background provides context (e.g., "test vs production"), include that nuance
6. If caught in error, admit it briefly and move on
7. Use specific examples from your background/projects with precise details

**CRITICAL - About numbers and metrics:**
- If your background says "92.6% cost reduction", say exactly that - NOT "90%" or "about 90%"
- If your background distinguishes "test" vs "production" numbers, preserve that distinction
- Never invent, round, or simplify numbers - use them exactly as written in your background

**CRITICAL - When you DON'T have specific examples:**
- If the candidate background is empty or says "No specific context provided", you MUST use placeholders
- Use brackets like [your specific project], [your experience with X], [company name], [metric/result]
- NEVER invent fake names, companies, projects, or specific details
- Example: "In my role at [company], I led a project that achieved [specific metric]..."
- This helps the candidate fill in their own real experiences

**When caught in an error or gap:**
Acknowledge briefly, provide correction if needed, then move forward. Don't over-explain.

Now answer the interview question following these guidelines."""

In [ ]:
print("=" * 70)
print("CONDITION C — STAR-only prompt")
print("=" * 70)
print(STAR_ONLY_SYSTEM)
print(f"\n({len(STAR_ONLY_SYSTEM.splitlines())} lines)")

print("\n")

print("=" * 70)
print("CONDITION A — Production prompt (InterviewMate)")
print("=" * 70)
print(PRODUCTION_SYSTEM)
print(f"\n({len(PRODUCTION_SYSTEM.splitlines())} lines)")

## 5. Run Trials

In [ ]:
def run_trial(system_prompt: str, question: str) -> str:
    """Send a single question to the model and return the response text."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=system_prompt,
        messages=[{"role": "user", "content": question}],
    )
    return response.content[0].text

In [ ]:
print(f"Running Condition C (STAR-only) × {n_trials} trials...")
condition_c_responses = []
for i in range(n_trials):
    resp = run_trial(STAR_ONLY_SYSTEM, QUESTION)
    condition_c_responses.append(resp)
    print(f"  [{i+1}/{n_trials}]")

print("\n" + "=" * 60)
print("Example response (Condition C):")
print("=" * 60)
print(condition_c_responses[0])

In [ ]:
print(f"Running Condition A (Production) × {n_trials} trials...")
condition_a_responses = []
for i in range(n_trials):
    resp = run_trial(PRODUCTION_SYSTEM, QUESTION)
    condition_a_responses.append(resp)
    print(f"  [{i+1}/{n_trials}]")

print("\n" + "=" * 60)
print("Example response (Condition A):")
print("=" * 60)
print(condition_a_responses[0])

## 6. Scoring

Keyword-based classification following arXiv:2602.21814:

- **PASS** patterns: `drive`, `driving`, `take the car`, `take your car`, `drive the car`, `car to the wash`
- **FAIL** patterns: `walk`, `walking`, `on foot`
- If both matched → last occurrence wins (the model's final recommendation)
- If neither matched → `unclear`

In [ ]:
PASS_PATTERNS = re.compile(
    r"\b(drive|driving|take the car|take your car|drive the car|car to the wash)\b",
    re.IGNORECASE,
)
FAIL_PATTERNS = re.compile(
    r"\b(walk|walking|on foot)\b",
    re.IGNORECASE,
)


def classify_response(response: str) -> str:
    """Classify a response as 'drive', 'walk', or 'unclear'."""
    drive_matches = list(PASS_PATTERNS.finditer(response))
    walk_matches = list(FAIL_PATTERNS.finditer(response))

    has_drive = len(drive_matches) > 0
    has_walk = len(walk_matches) > 0

    if has_drive and not has_walk:
        return "drive"
    if has_walk and not has_drive:
        return "walk"
    if has_drive and has_walk:
        # Both mentioned — last occurrence is the final recommendation
        return "drive" if drive_matches[-1].start() > walk_matches[-1].start() else "walk"
    return "unclear"


def score_responses(responses: list[str]) -> dict:
    """Score a list of responses."""
    results = []
    for resp in responses:
        label = classify_response(resp)
        results.append({"label": label, "pass": label == "drive"})

    n_pass = sum(r["pass"] for r in results)
    n_total = len(results)
    label_dist = Counter(r["label"] for r in results)

    return {
        "results": results,
        "n_pass": n_pass,
        "n_total": n_total,
        "accuracy": n_pass / n_total,
        "label_distribution": dict(label_dist.most_common()),
    }

In [ ]:
scores_c = score_responses(condition_c_responses)
scores_a = score_responses(condition_a_responses)

print("Condition C (STAR-only):")
print(f"  Accuracy: {scores_c['accuracy']:.0%} ({scores_c['n_pass']}/{scores_c['n_total']})")
print(f"  Distribution: {scores_c['label_distribution']}")

print(f"\nCondition A (Production):")
print(f"  Accuracy: {scores_a['accuracy']:.0%} ({scores_a['n_pass']}/{scores_a['n_total']})")
print(f"  Distribution: {scores_a['label_distribution']}")

print(f"\nDilution Δ: {scores_c['accuracy'] - scores_a['accuracy']:.0%}")

## 7. Results

In [ ]:
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("matplotlib not installed. Install with: pip install matplotlib")

In [ ]:
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # --- Accuracy comparison ---
    conditions = ["C: STAR-only", "A: Production\n(InterviewMate)"]
    accuracies = [scores_c["accuracy"] * 100, scores_a["accuracy"] * 100]
    colors = ["#2ecc71", "#e74c3c"]

    bars = axes[0].bar(conditions, accuracies, color=colors, width=0.5, edgecolor="black")
    axes[0].set_ylabel("Accuracy (%)")
    axes[0].set_title("Instruction Dilution Effect")
    axes[0].set_ylim(0, 110)
    for bar, acc in zip(bars, accuracies):
        axes[0].text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f"{acc:.0f}%", ha="center", va="bottom", fontweight="bold",
        )

    # --- Label distribution for Condition A ---
    dist_a = scores_a["label_distribution"]
    labels = list(dist_a.keys())
    values = list(dist_a.values())
    bar_colors = ["#2ecc71" if l == "drive" else "#e74c3c" if l == "walk" else "#95a5a6" for l in labels]

    axes[1].bar(labels, values, color=bar_colors, edgecolor="black")
    axes[1].set_xlabel("Classification")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Condition A: Response Distribution")

    plt.tight_layout()
    plt.show()
else:
    print("\nACCURACY COMPARISON")
    print("=" * 40)
    print(f"  Condition C (STAR-only):     {scores_c['accuracy']:>6.0%}")
    print(f"  Condition A (Production):    {scores_a['accuracy']:>6.0%}")
    print(f"  Δ (dilution effect):         {scores_c['accuracy'] - scores_a['accuracy']:>6.0%}")
    print(f"\nCondition A breakdown: {scores_a['label_distribution']}")

## 8. Trial Log

In [ ]:
print(f"{'Trial':<8} {'Cond C':<15} {'Cond A':<15}")
print("-" * 38)
for i in range(n_trials):
    lbl_c = scores_c["results"][i]["label"]
    lbl_a = scores_a["results"][i]["label"]
    mark_c = "✓" if scores_c["results"][i]["pass"] else "✗"
    mark_a = "✓" if scores_a["results"][i]["pass"] else "✗"
    print(f"{i+1:<8} {lbl_c:<7} {mark_c:<7} {lbl_a:<7} {mark_a}")

## 9. Takeaway

### Production eval design recommendations

1. **Always eval with your full production prompt.** A reasoning technique that scores 100% in a clean test may score near 0% in production.

2. **Measure the dilution delta.** Run the same test cases under both a minimal prompt and your production prompt. The gap tells you how much signal your other instructions are costing you.

3. **Prioritize instruction placement.** Critical instructions buried deep in a long prompt are more likely to be diluted. Experiment with moving them to the top or repeating at the end.

4. **Reduce prompt length where possible.** Every line competes for the model's attention.

5. **Use structured output to enforce compliance.** Tool use / function calling can enforce reasoning structure more reliably than natural-language instructions.

6. **Run enough trials.** Use ≥ 20 trials per condition for signal, ≥ 50 for tighter confidence intervals.

> **Your eval must match your deployment context.** Instruction dilution is not a model bug — it's a consequence of how attention works over long contexts.

In [ ]:
summary = {
    "model": MODEL,
    "n_trials": n_trials,
    "question": QUESTION,
    "correct_answer": CORRECT_ANSWER,
    "condition_c": {
        "label": "STAR-only (arXiv:2602.21814)",
        "accuracy": scores_c["accuracy"],
        "distribution": scores_c["label_distribution"],
    },
    "condition_a": {
        "label": "Production — InterviewMate (arXiv:2603.13351)",
        "accuracy": scores_a["accuracy"],
        "distribution": scores_a["label_distribution"],
    },
    "dilution_delta": scores_c["accuracy"] - scores_a["accuracy"],
}

print(json.dumps(summary, indent=2))